In [ ]:
import pandas as pd
import numpy as np
import re
import emoji
import joblib
import torch
import string
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score, precision_score, recall_score, f1_score
from torch.utils.data import DataLoader, Dataset
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
import nltk
from nltk.corpus import stopwords
nltk.download('stopwords')

In [ ]:
train_data = pd.read_csv("train.csv")

In [ ]:
import pandas as pd
dataset = pd.read_csv('train.csv')
print("Dataset shape:", dataset.shape)


In [ ]:
'''def remove_URL(text):
    text=str(text)
    url = re.compile(r'https?://\S+|www\.\S+')
    return url.sub(r'', text)


def remove_emoji(text):
    emoji_pattern = re.compile(
        '['
        u'\U0001F600-\U0001F64F'
        u'\U0001F300-\U0001F5FF'
        u'\U0001F680-\U0001F6FF'
        u'\U0001F1E0-\U0001F1FF'
        u'\U00002702-\U000027B0'
        u'\U000024C2-\U0001F251'
        ']+',
        flags=re.UNICODE)
    return emoji_pattern.sub(r'', text)

def remove_mentions(text):
    ment = re.compile(r"(@[A-Za-z0-9]+)")
    return ment.sub(r'', text)


def remove_html(text):
    html = re.compile(r'<.*?>|&([a-z0-9]+|#[0-9]{1,6}|#x[0-9a-f]{1,6});')
    return re.sub(html, '', text)


def remove_punct(text):
    table = str.maketrans('', '', string.punctuation)
    return text.translate(table)'''

In [ ]:
import string
import urllib.parse
def remove_URL(text):
    text = str(text)
    url = re.compile(r'https?://\S+|www\.\S+')
    return url.sub(r'', text)

def retain_emoji(text):
    text = emoji.demojize(text, delimiters=("", " "))
    return text

def remove_mentions(text):
    ment = re.compile(r"(@[A-Za-z0-9_]+)")
    return ment.sub(r'', text)

def remove_html(text):
    html = re.compile(r'<.*?>|&([a-z0-9]+|#[0-9]{1,6}|#x[0-9a-f]{1,6});')
    return re.sub(html, '', text)

def remove_punct(text):
    table = str.maketrans('', '', string.punctuation)
    return text.translate(table)

def preprocess_text(text):
    text = remove_URL(text)
    text = retain_emoji(text)
    text = remove_mentions(text)
    text = remove_html(text)
    text = remove_punct(text)
    return text.strip()

In [ ]:
train_data['clean_text'] = train_data['tweets'].apply(lambda x: preprocess_text(x))
train_data['clean_text'] = train_data['clean_text'].apply(lambda x: x.lower())
cleaned = train_data['clean_text'].tolist()

for i,text in enumerate(cleaned):
    splits = text.split()
    splits = [word for word in splits if word not in set(nltk.corpus.stopwords.words('english'))]
    cleaned[i]=' '.join(splits)

train_data['clean_text']=cleaned

In [ ]:
from sklearn.preprocessing import LabelEncoder


label_encoder = LabelEncoder()
y_train = label_encoder.fit_transform(train_data['class'].values)

In [ ]:
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()
y_train = label_encoder.fit_transform(train_data['class'].values)


class_labels = train_data['class'].unique()


for label, encoded_label in zip(class_labels, label_encoder.transform(class_labels)):
    print(f"Original Label: {label}, Encoded Label: {encoded_label}")

In [ ]:

train_texts, val_texts, train_labels, val_labels = train_test_split(
    train_data['tweets'].tolist(),
    train_data['class'].tolist(),
    test_size=0.2,
    random_state=42
)

print(f"\nTraining samples: {len(train_texts)}")
print(f"Validation samples: {len(val_texts)}")

In [ ]:
def scores(y_test, y_predicted):

  accuracy = accuracy_score(y_test, y_predicted)
  print('Accuracy: %f' % accuracy)

  precision = precision_score(y_test, y_predicted, average='weighted')
  print('Precision: %f' % precision)

  recall = recall_score(y_test, y_predicted, average='weighted')
  print('Recall: %f' % recall)

  f1 = f1_score(y_test, y_predicted, average='weighted')
  print('F1 score: %f' % f1)
  return precision, recall, f1

In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from nltk.corpus import stopwords
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
import nltk



nltk.download('stopwords')



train_data = pd.read_csv('train.csv')

x = train_data['tweets'].tolist()

stopWords = set(stopwords.words('english'))

filtered_tweets = []
for tweet in x:
    words = tweet.split()
    words_filtered = [word for word in words if word.lower() not in stopWords]
    filtered_tweets.append(" ".join(words_filtered))

vectorizer = CountVectorizer(lowercase=True)
features = vectorizer.fit_transform(filtered_tweets)

features_nd = features.toarray()

label_encoder = LabelEncoder()
y = label_encoder.fit_transform(train_data['class'].values)

x_train, x_test, y_train, y_test = train_test_split(features_nd, y, train_size=0.80, random_state=1234)

print(x_train.shape)
print(y_train.shape)

log_model = LogisticRegression(max_iter=1000)
log_model.fit(x_train, y_train)

y_pred = log_model.predict(x_test)

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average='weighted')
recall = recall_score(y_test, y_pred, average='weighted')
f1 = f1_score(y_test, y_pred, average='weighted')

print(f'Accuracy: {accuracy}')
print(f'Precision: {precision}')
print(f'Recall: {recall}')
print(f'F1 Score: {f1}')

In [ ]:
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score


dataset = pd.read_csv("/train.csv", sep=',')


print("Dataset columns:", dataset.columns)
print(dataset.head())


dataset = dataset.dropna(subset=['tweets', 'class'])


print(f"Dataset shape after cleaning: {dataset.shape}")


text_column_name = 'tweets'
label_column_name = 'class'


tokenizer = Tokenizer(num_words=30000, split=' ')
tokenizer.fit_on_texts(dataset[text_column_name].values)


X = tokenizer.texts_to_sequences(dataset[text_column_name].values)


X = pad_sequences(X, padding='post', maxlen=35)


print(f"Shape of tokenized and padded input features: {X.shape}")


Y = pd.get_dummies(dataset[label_column_name]).values


print(f"Shape of one-hot encoded labels: {Y.shape}")


X_train, X_test, Y_train, Y_test = train_test_split(X, Y, train_size=0.8, random_state=42)


print(f"X_train shape: {X_train.shape}, Y_train shape: {Y_train.shape}")
print(f"X_test shape: {X_test.shape}, Y_test shape: {Y_test.shape}")


train_dataset = tf.data.Dataset.from_tensor_slices((X_train, Y_train))
test_dataset = tf.data.Dataset.from_tensor_slices((X_test, Y_test))


train_dataset = train_dataset.shuffle(buffer_size=10000).batch(64)
test_dataset = test_dataset.batch(64)


model = tf.keras.Sequential([

    tf.keras.layers.Embedding(input_dim=30000, output_dim=128, input_length=35),


    tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(256, return_sequences=True)),


    tf.keras.layers.Dropout(0.5),


    tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(128)),


    tf.keras.layers.Dense(128, activation='relu'),


    tf.keras.layers.Dropout(0.5),


    tf.keras.layers.Dense(2, activation='softmax')
])


model.build(input_shape=(None, 35))


model.summary()


model.compile(loss='categorical_crossentropy',
              optimizer=tf.keras.optimizers.Adam(1e-4),
              metrics=['accuracy'])


history = model.fit(train_dataset, epochs=10)


results = model.evaluate(test_dataset)
print('Test loss, test accuracy:', results)


y_pred = model.predict(X_test)


y_pred_classes = y_pred.argmax(axis=1)
y_test_classes = Y_test.argmax(axis=1)


print(f'Accuracy: {accuracy_score(y_test_classes, y_pred_classes)}')
print(f'Precision: {precision_score(y_test_classes, y_pred_classes, average="weighted")}')
print(f'Recall: {recall_score(y_test_classes, y_pred_classes, average="weighted")}')
print(f'F1 Score: {f1_score(y_test_classes, y_pred_classes, average="weighted")}')


In [ ]:
import tensorflow as tf
import numpy as np
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences




sentences = [
    "I love getting mail saying that my medical &amp; personal information has been compromised. #sarcasm",
    "This has been the worst experience of my life.",
    "Oh great, another Monday morning. Just what I needed.",
    "The fire station burned down last night.",
    "I went to the store to buy some groceries."
]


tokenizer = Tokenizer(num_words=10000, oov_token="<OOV>")
tokenizer.fit_on_texts(sentences)
sequences = tokenizer.texts_to_sequences(sentences)
padded_sequences = pad_sequences(sequences, maxlen=100, padding='post', truncating='post')


predictions = model.predict(padded_sequences)


class_labels = ['irony', 'sarcasm']


print("Prediction Results:")
for i, sentence in enumerate(sentences):
    predicted_class_index = np.argmax(predictions[i])
    predicted_class = class_labels[predicted_class_index]
    print(f"Input Sentence: {sentence}")
    print(f"Prediction: {predicted_class} (Probabilities: {predictions[i]})")

In [ ]:
import os


save_dir = "/content/saved_models"


if not os.path.exists(save_dir):
    os.makedirs(save_dir)


model_path = os.path.join(save_dir, "sarcasm_and_irony_model.h5")


model.save(model_path)

print(f"Model saved at: {model_path}")
